# Delivery Performance, Delay Risk, and Logistics Efficiency Analysis

This notebook performs the full EDA and logistics diagnostics for the APL Logistics project.

**Important:** The supplied dataset has no explicit date column. The notebook therefore focuses on delivery-gap, shipping-mode, regional, market and customer-segment analysis rather than synthetic date trends.


In [ ]:
!pip -q install pandas numpy scipy matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.colab import files
from IPython.display import display

print("Libraries loaded.")


## 1. Upload and load the dataset

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))

# Handle common encodings.
for encoding in ["utf-8", "cp1252", "latin1"]:
    try:
        df = pd.read_csv(file_name, encoding=encoding)
        print("Loaded with encoding:", encoding)
        break
    except UnicodeDecodeError:
        continue

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Data types:")
df.info()

print("\nMissing values by column:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", df.duplicated().sum())

core_cols = [
    "Days for shipping (real)", "Days for shipment (scheduled)",
    "Late_delivery_risk", "Delivery Status",
    "Shipping Mode", "Order Region", "Market", "Customer Segment"
]

print("\nMissing values in core analytical fields:")
print(df[core_cols].isna().sum())


## 2. Delivery gap and classification

In [ ]:
df["Delay_Gap_Days"] = (
    df["Days for shipping (real)"]
    - df["Days for shipment (scheduled)"]
)

df["Delivery_Class"] = np.select(
    [
        df["Delay_Gap_Days"] > 0,
        df["Delay_Gap_Days"] < 0
    ],
    ["Delayed", "Early"],
    default="On-time"
)

df["Is_Cancelled"] = df["Delivery Status"].eq("Shipping canceled")
df["Is_Delayed"] = df["Delay_Gap_Days"] > 0
df["Is_On_Time_or_Early"] = df["Delay_Gap_Days"] <= 0

delivered = df.loc[~df["Is_Cancelled"]].copy()

print("Delay gap statistics:")
print(delivered["Delay_Gap_Days"].describe().round(3))

print("\nDelivery class:")
print(df["Delivery_Class"].value_counts())


## 3. Overall delivery performance

In [ ]:
on_time_rate = delivered["Is_On_Time_or_Early"].mean() * 100
delay_rate = delivered["Is_Delayed"].mean() * 100
avg_gap = delivered["Delay_Gap_Days"].mean()
avg_delay_delayed_only = delivered.loc[delivered["Is_Delayed"], "Delay_Gap_Days"].mean()
risk_rate = df["Late_delivery_risk"].mean() * 100

print(f"On-time / early delivery rate: {on_time_rate:.2f}%")
print(f"Calculated delayed-delivery rate: {delay_rate:.2f}%")
print(f"Average delivery gap: {avg_gap:.3f} days")
print(f"Average delay among delayed deliveries: {avg_delay_delayed_only:.3f} days")
print(f"Late_delivery_risk rate: {risk_rate:.2f}%")

status = df["Delivery Status"].value_counts().rename_axis("Delivery Status").reset_index(name="Orders")
status["Share_%"] = status["Orders"] / len(df) * 100
display(status)

plt.figure(figsize=(8, 5))
sns.barplot(data=status, x="Delivery Status", y="Orders")
plt.xticks(rotation=20)
plt.title("Delivery Status Distribution")
plt.tight_layout()
plt.show()


## 4. Shipping mode efficiency

In [ ]:
mode = (
    delivered.groupby("Shipping Mode")
    .agg(
        Orders=("Delay_Gap_Days", "size"),
        Delayed=("Is_Delayed", "sum"),
        On_Time_or_Early=("Is_On_Time_or_Early", "sum"),
        Avg_Delay_Gap_Days=("Delay_Gap_Days", "mean")
    )
    .reset_index()
)
mode["Delay_Rate"] = mode["Delayed"] / mode["Orders"] * 100
mode["Efficiency_Index"] = mode["On_Time_or_Early"] / mode["Orders"] * 100

display(mode.round(3).sort_values("Delay_Rate"))

plt.figure(figsize=(8, 5))
sns.barplot(data=mode, x="Shipping Mode", y="Delay_Rate")
plt.xticks(rotation=15)
plt.title("Delay Rate by Shipping Mode")
plt.ylabel("Delay rate (%)")
plt.show()

table = pd.crosstab(delivered["Shipping Mode"], delivered["Delivery_Class"])
chi2, p, dof, expected = stats.chi2_contingency(table)
n = table.values.sum()
v = np.sqrt((chi2/n) / min(table.shape[0]-1, table.shape[1]-1))

print(f"Chi-square = {chi2:.3f}")
print(f"p-value = {p:.6g}")
print(f"Cramer's V = {v:.4f}")


## 5. Regional and market diagnostics

In [ ]:
def group_kpis(data, column, baseline_delay_rate):
    out = (
        data.groupby(column)
        .agg(
            Orders=("Delay_Gap_Days", "size"),
            Delayed=("Is_Delayed", "sum"),
            On_Time_or_Early=("Is_On_Time_or_Early", "sum"),
            Avg_Delay_Gap_Days=("Delay_Gap_Days", "mean")
        )
        .reset_index()
    )
    out["Delay_Rate"] = out["Delayed"] / out["Orders"] * 100
    out["On_Time_or_Early_Rate"] = out["On_Time_or_Early"] / out["Orders"] * 100
    out["Regional_Delay_Index"] = out["Delay_Rate"] / baseline_delay_rate * 100
    return out.sort_values("Delay_Rate", ascending=False)

region = group_kpis(delivered, "Order Region", delay_rate)
market = group_kpis(delivered, "Market", delay_rate)

print("Top regions:")
display(region.head(15).round(3))

print("Markets:")
display(market.round(3))


## 6. Customer segment diagnostics

In [ ]:
segment = group_kpis(delivered, "Customer Segment", delay_rate)
display(segment.round(3))

table = pd.crosstab(delivered["Customer Segment"], delivered["Delivery_Class"])
chi2, p, dof, expected = stats.chi2_contingency(table)
n = table.values.sum()
v = np.sqrt((chi2/n) / min(table.shape[0]-1, table.shape[1]-1))

print(f"Customer Segment chi-square = {chi2:.3f}")
print(f"p-value = {p:.6g}")
print(f"Cramer's V = {v:.4f}")


## 7. Late_delivery_risk versus calculated delivery class

In [ ]:
risk_table = pd.crosstab(df["Late_delivery_risk"], df["Delivery_Class"])
display(risk_table)

print("Percentage within each Late_delivery_risk group:")
display(pd.crosstab(
    df["Late_delivery_risk"],
    df["Delivery_Class"],
    normalize="index"
).mul(100).round(3))


## 8. Final interpretation checklist

- Report the delivery gap and `Late_delivery_risk` as separate indicators.
- Exclude cancelled orders from completed-delivery timing KPIs.
- Use effect sizes together with p-values.
- Do not infer causation from shipping-mode or regional association.
- The supplied dataset has no date column, so do not create a synthetic date trend.
